# Experiment v6 — Calibration + Isolation Phase (Dell PowerEdge R7615)

Run the cells in order. **Section 3 (C0 calibration) comes before the main run and its output
decides the budget levels used in section 5** — see `analysis/EXPERIMENT_V6_PLAN.md`.

What is new in v6:
- **Greedy initial sequence** (standard EMCTS's default), shared across all 12 configs per (scenario, visit).
- **The budget bounds SEARCH only** — greedy initialization is measured and excluded; bridge burning,
  including the crossover tournament, stays inside the budget.
- **50 visits** per scenario (v5: 100).
- **Realized spend logged per turn** (units, advances, copies, evals, init cost, wall-clock).

How this notebook treats the server (see `analysis/V6_SERVER_RUNBOOK.md` for the full rationale):
- **Threads are capped at PHYSICAL cores, never the SMT count.** `ServerProfile` reads
  `/proc/cpuinfo`, subtracts a small reserve for GC/JIT/OS, and that ceiling binds every pool in the
  pipeline — portfolio build, thread calibration and the isolation phase alike.
- **The thread count below that ceiling is measured, not guessed**, per budget level, from *sustained*
  throughput — each sweep point runs twice so a machine that throttles under load is visible.
- **A load governor lowers concurrency during the run** if the heap comes under pressure, and raises it
  again when it clears.
- **Every stage is restartable.** Ctrl-C / SIGTERM drains cleanly; a hard kill costs at most one turn
  per worker. Re-running the same cell resumes.
- **Only one writer per output tree.** Each job takes a heartbeat lock and refuses to start beside a
  live one, so "re-run the cell to resume" can never silently double-write the dataset.

In [ ]:
# EDIT to the absolute path of the Tribes-master directory on the server:
TRIBES = '/root/Bachelor_Thesis/Tribes-master'
%cd $TRIBES

## 0. Environment check
Java, CPU topology, memory, disk. Two things to confirm before anything else:

1. **Physical cores vs threads.** `lscpu` reports `Core(s) per socket` × `Socket(s)` as the physical
   count and `Thread(s) per core` as SMT. On a fully-populated R7615 that is 96 physical / 192 logical.
   The pipeline will use the physical number minus a reserve, and says so in its own banner.
2. **Nothing else is running.** The C0 calibration in section 3 is a *timing* experiment on this exact
   machine — any competing load makes its numbers wrong rather than merely noisy.

In [ ]:
!java -version 2>&1 | head -3
print('--- CPU topology ---')
!lscpu | egrep 'Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket|Socket\(s\)|NUMA node\(s\)|MHz'
print('--- memory / disk ---')
!free -g | head -2
!df -h . | tail -1
print('--- power/thermal governor (affects sustained clocks) ---')
!cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_governor 2>/dev/null || echo '(cpufreq not exposed)'
print('--- other java processes (should be none before C0) ---')
!pgrep -af java || echo '(none - good)'
print('--- v6 run locks currently held ---')
!find test_results/v6 -name '_running.lock' -exec sh -c 'echo "$1: $(cat $1)"' _ {} \; 2>/dev/null || true

## 1. Build
Recompiles everything from `src/`. The source list is regenerated each time, so the new v6 classes
(`ServerProfile`, `RunGuard`, `LoadGovernor`, the rewritten `ThreadCalibrationV6` and
`BuildV6Portfolio`) are picked up automatically.

In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
# --release 11 pins the language level AND the API set to the server's JDK, so anything newer is a
# compile error on the developer's machine rather than a surprise here. (It already caught one:
# Thread.stop() is final up to JDK 22 and removed in 23, so a HeapSampler.stop() compiled fine on a
# JDK 26 laptop and failed on this server.)
!javac --release 11 -d out -cp "lib/*" @_srcs.txt && echo 'BUILD OK' || echo 'BUILD FAILED'

## 1b. Machine policy — what the pipeline will actually do
Prints the ceiling and the thread ladder this machine gets, without running anything. Check it against
section 0 before committing to a multi-day run.

To override (a shared machine, or an explicit core count when `/proc/cpuinfo` is unhelpful):

- `-Dv6.maxThreads=N`  — hard cap on workers for every stage
- `-Dv6.reserveCores=N` — cores held back for GC/JIT/OS (default ~3%, min 1, max 4)

Both are also readable from the environment as `V6_MAX_THREADS` / `V6_RESERVE_CORES`.

In [ ]:
!java -cp "out:lib/*" -Xmx4g core.data.exp.ServerProfile

## 2. Smoke test (~30 s) — pipeline correctness
Verifies FastGameStateIO round-trip, the shared **greedy** initial sequence across agents AND budgets,
folder layout, CSV schema, operator telemetry, and the two budget-boundary properties:

- **init OUT of the budget**: every agent reports a non-zero initialization cost while its search spend
  still reaches the full budget (if init were charged, search would fall short by exactly that amount);
- **bridge burning IN the budget**: all agents perform 20 burns, the crossover agents stay at the budget
  rather than exceeding it, and they buy strictly fewer distinct sequences than EMCTS — the tournaments
  are paid for out of the same pot.

If this fails, do NOT launch C0 or the full run.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV6

## 3. Portfolio (~1 h, parallel + restartable)
Builds the 200-scenario portfolio if it is not already there. Source games run continuously across
workers and each finished game is journalled, so **an interrupted build resumes instead of replaying
every game from attempt 1** — re-run this cell after any interruption.

The build takes the machine's worker ceiling automatically; pass a number instead of `auto` to force
one. Heap: source games are small, 64 GB is generous.

In [ ]:
!java -Xmx64g -cp "out:lib/*" core.data.exp.BuildV6Portfolio test_results/v6/_portfolio auto

## 3b. Compute-unit weights — calibrate this machine (one cell, ~3 min)
A compute-unit is `advances + W_COPY*copies + W_EVAL*evals`. Those weights are the relative wall-clock
costs of the three operations, so they belong to the **machine**, not to the code — the ones compiled
in came from the dev laptop.

Run the cell. It warms up, repeats the measurement several times, prints a
**IS THIS A FAIR BENCHMARK?** block with a pass/fail per criterion, and — only if every check passes —
writes `test_results/v6/_calibration/compute_weights.properties`, which **every later job in this
notebook loads automatically**. Nothing to copy, nothing to rebuild.

What the checks mean:

| check | why it matters |
|---|---|
| enough advances / copies / evals | a per-operation mean divided by too few samples is noise |
| CV across repetitions ≤ 10% | if the same measurement moves between repeats, the machine was not quiet |
| weights plausible (each < 1 advance) | catches a broken or mis-attributed timer |
| load average idle | a busy machine inflates copies and evals more than advances — i.e. it biases exactly the ratio being measured |

If it FAILS it writes nothing and the pipeline keeps its current weights, so a bad measurement can
never silently reprice the experiment. The usual cause is other load: stop it and re-run.

**Do this before section 4 (C0) and before the run.** Changing the weights changes what a unit *is*,
so the budget levels have to be derived afterwards, not before — and never change them mid-experiment.

In [ ]:
!java -Xmx16g -cp "out:lib/*" core.data.exp.ComputeCalibrateV6

In [ ]:
# What is now in force, and where it came from. Re-run any time to confirm.
WEIGHTS = 'test_results/v6/_calibration/compute_weights.properties'
import os
print(open(WEIGHTS).read() if os.path.exists(WEIGHTS)
      else 'no calibration file yet - the pipeline is using the compiled defaults')

### 3c. Is the budget fair between the agents, and does each operator fire?
`BudgetFairnessProbe` measures each configuration's realized operation mix and re-prices all of them
under alternative weights. If the configurations spend alike, a wrong weight rescales everyone together
and cancels out of the paired comparison; the probe reports the worst-case distortion in percentage
points, so fairness is a measurement rather than a hope. It also reports budget adherence and whether
each crossover operator actually fires at that budget.

In [ ]:
# Read the VERDICT block of each. Budgets come from RunExperimentV6.DEFAULT_COMPUTE_LEVELS.
LEVELS = (40000, 80000, 400000)
for cu in LEVELS:
    print()
    print('=' * 70)
    print('  budget', cu, 'cu')
    print('=' * 70)
    !java -Xmx32g -cp "out:lib/*" core.data.exp.BudgetFairnessProbe test_results/v6/_portfolio test_results/v6/_calibration {cu} 12

### 3d. Budget floor — why the smallest level is what it is
The crossover bridge-burn tournament is a 3-level bracket over the phase's top **eight** elites
(8 → 4 → 2 → 1). The code runs it whenever the pool holds at least **two**, so below a certain budget a
burn either skips the tournament entirely — falling back to promoting the best child, which is what
plain EMCTS does — or breeds from a pool of 2 or 3 and is not the operator being studied. Neither
announces itself in the results.

This probe sweeps candidate budgets and reports, for **CrossoverBB and CrossoverCombo separately**
(BB is the binding case — it has no expansion crossover feeding its pool), the share of the 20 burns
that ran a real tournament, the mean bracket size out of 8, and how many new nodes are added per phase
between consecutive burns. Re-run it if the compute weights change, since a unit will mean something
different.

In [ ]:
!java -Xmx16g -cp "out:lib/*" core.data.exp.BudgetFloorProbe test_results/v6/_portfolio test_results/v6/_calibration 26800,33500,40000,47000,60000 10

## 4. C0 — thinking-time calibration (~2.6 h, SINGLE-THREADED)
**Run this before section 5 and read its output.** Gives each of the 12 configs a fixed amount of
SEARCH time (500 / 1000 / 5000 ms, each a separate run) and records what it actually does: advances,
copies, evals, weighted units, and the greedy-initialization cost separately.

It answers four questions:
1. the units→seconds conversion the paper reports for this reference machine;
2. whether that conversion is **agent-invariant** — which decides whether v6 uses one shared
   compute-unit budget or per-agent budgets calibrated to reference time;
3. whether it is invariant across scenario source and branching factor (the H5 confound check);
4. how large `InitUnits` actually is, and therefore whether the smallest budget level needs raising.

Single-threaded **on purpose**: this machine is the reference machine, so the run must be the only
load on it. Leave it otherwise idle — do not start section 5 alongside it.

Interrupted? Re-run the same cell: fully-measured targets are kept and only the interrupted one is
redone. A target that is only half-measured is discarded rather than mixed with a second run's.

In [ ]:
# args: [portfolioDir] [outDir] [targetsMsCsv] [perSource] [reps] [warmup]
!nohup java -Xmx32g -cp "out:lib/*" core.data.exp.CalibrationRunnerV6 \
    test_results/v6/_portfolio test_results/v6/_calibration 500,1000,5000 12 5 40 \
    > v6_calibration.log 2>&1 &
print('C0 started in background -> v6_calibration.log')

In [ ]:
# C0 progress + the per-config summary once it finishes (re-run to refresh).
!echo "rows so far : $(($(wc -l < test_results/v6/_calibration/calibration.csv 2>/dev/null || echo 1)-1))"
!pgrep -af CalibrationRunnerV6 || echo '(C0 not running - finished or not started)'
!tail -n 40 v6_calibration.log 2>/dev/null || echo '(no log yet)'

### 4b. Read C0 → set the v6 budget levels
The table below is the decision. Read **units/ms across configs at a fixed target**: if it is flat, one
shared compute-unit budget is fair and you keep `RunExperimentV6.DEFAULT_COMPUTE_LEVELS`; if it varies
systematically by agent, budgets must be calibrated per agent instead. Also check `InitUnits` against
the smallest level (40,000): initialization is excluded from the budget, but if it is a large fraction
of it, the turn is spending more outside the measured budget than inside it, which is worth reporting.

Note the smallest level is set by `BudgetFloorProbe` (section 3d), not by C0 — C0 answers "what does
an agent do in T ms", which is a different question from "is the operator structurally able to run".

In [ ]:
import pandas as pd
c = pd.read_csv('test_results/v6/_calibration/calibration.csv')
c['Config'] = c.Config + c.Ratio.astype(str).radd('@r').where(c.Ratio != 'NA', '')
c['UnitsPerMs'] = c.SearchUnits / c.SearchMillis.clip(lower=1)

print('=== units/ms by config x target (is it flat across configs?) ===')
display(c.pivot_table(index='Config', columns='TargetMillis', values='UnitsPerMs', aggfunc='mean').round(2))

print('=== units/ms by scenario source (the H5 confound check) ===')
display(c.pivot_table(index='Source', columns='TargetMillis', values='UnitsPerMs', aggfunc='mean').round(2))

print('=== greedy-init cost by source, and as a share of the smallest planned level ===')
init = c.groupby('Source').InitUnits.agg(['mean', 'median', 'max']).round(0)
init['share_of_floor'] = (init['mean'] / 40000).round(3)   # vs the smallest v6 level
display(init)

print('=== stability across repetitions (CV should be small on a quiesced machine) ===')
cv = c.groupby(['TargetMillis', 'Config']).SearchUnits.agg(['mean', 'std'])
cv['cv'] = (cv['std'] / cv['mean']).round(4)
display(cv.cv.groupby('TargetMillis').describe()[['mean', 'max']].round(4))

print('=== suggested per-agent budgets, if the per-agent route is taken ===')
display(c.pivot_table(index='Config', columns='TargetMillis', values='SearchUnits', aggfunc='mean').round(0))

## 5. Run v6 (background + log) — also the RESUME cell
Launches the full run detached so it survives a disconnect. **Re-run this same cell to resume** after a
server restart — finished work is skipped, an interrupted config continues per (scenario, visit), and
the run refuses to start if a previous one is still alive.

Args: `[threads|auto] [computeUnitsCsv]`. `auto` (default) runs the pre-run phases: portfolio build →
startup verify → **greedy-init invariance check** → **per-budget thread calibration** → isolation.

The budget levels default to **40,000 / 80,000 / 400,000** (floor, 2x, 10x), derived by
`BudgetFloorProbe` in section 3d — see `V6_SERVER_RUNBOOK.md` §5d. Override by passing them
explicitly, e.g. `auto 40000,80000,400000`. If C0 or a re-calibration of the compute weights moves the
floor, re-run section 3d and pass the new set.

Scale: 200 scenarios × 50 visits × 12 configs × 3 levels = 360,000 turns (half of v5), but each turn
is ~3x the old budget, so plan for roughly 3x v5's isolation-phase wall-clock.

Heap: give the JVM the machine's RAM minus what the OS needs. `-Xmx750g` suits a 768 GB R7615; lower
it and the thread calibration will simply choose fewer workers. `-XX:+UseG1GC` and a generous
`-XX:MaxGCPauseMillis` suit a throughput job with large short-lived trees; the load governor reads
G1's post-collection occupancy to decide when to throttle.

In [ ]:
!nohup java -Xmx750g -XX:+UseG1GC -XX:MaxGCPauseMillis=500 \
    -cp "out:lib/*" core.data.exp.RunExperimentV6 auto > v6_run.log 2>&1 &
print('v6 started/resumed in background -> v6_run.log')

## 6. Progress (re-run to refresh)
Read-only view of the running job: pre-run checkpoints, the thread count chosen per budget level,
leaves finished, turns/min with a projected finish, live thread/CPU use (`NLWP` = OS threads, `%CPU` =
total across them), and the log tail. Governor lines in the tail show any throttling.

In [ ]:
import subprocess, time, os, glob, re

BASE, LEAVES, TURNS_PER_LEAF = 'test_results/v6', 36, 200 * 50

!echo "cores       : $(nproc) logical / $(lscpu | awk -F: '/^Core\(s\) per socket/{c=$2} /^Socket\(s\)/{s=$2} END{print c*s}') physical"
!echo "portfolio   : $(test -f test_results/v6/_portfolio/_portfolio_complete.txt && echo COMPLETE || echo building/incomplete)"
!echo "greedy-init : $(test -f test_results/v6/_greedy_init_ok.txt && echo ok || echo pending)"
!echo "calibration : $(test -f test_results/v6/_calibration/calibration.csv && echo present || echo 'MISSING - run section 4 first')"
!echo "lock        : $(cat test_results/v6/_running.lock 2>/dev/null || echo '(none - no run in progress)')"

# Thread count chosen per budget level (one file per level).
tuned = sorted(glob.glob(f'{BASE}/_tuned_threads_*cu.txt'))
if tuned:
    picks = ', '.join(f"{re.search(r'_([0-9]+)cu', f).group(1)}cu -> {open(f).read().strip()} threads"
                      for f in tuned)
    print('threads     :', picks)
else:
    print('threads     : pending (calibration has not finished a level yet)')

# Turn counts per leaf, from the telemetry file (one row per completed turn).
teles = sorted(glob.glob(f'{BASE}/**/turn_telemetry.csv', recursive=True))
total_turns = 0
print('--- per-leaf completed turns (full leaf = 10000) ---')
for f in teles:
    n = max(0, sum(1 for _ in open(f)) - 1)
    total_turns += n
    if n < TURNS_PER_LEAF:
        print(f'  {f.replace(BASE + "/", ""):50s} {n:6d}')
print(f'  TOTAL {total_turns} / {LEAVES * TURNS_PER_LEAF} turns '
      f'({100.0 * total_turns / (LEAVES * TURNS_PER_LEAF):.1f}%)')

# Throughput + ETA from the log's age.
log = 'v6_run.log'
if os.path.exists(log) and total_turns > 0:
    mins = max(1e-9, (time.time() - os.path.getctime(log)) / 60.0)
    rate = total_turns / mins
    left = LEAVES * TURNS_PER_LEAF - total_turns
    if rate > 0:
        print(f'  rate  {rate:.0f} turns/min   elapsed {mins/60:.1f} h   ETA {left / rate / 60:.1f} h')

print('--- run process: PID / threads (NLWP) / CPU% / elapsed ---')
!ps -o pid,nlwp,pcpu,etime,comm -C java 2>/dev/null || echo '(no java process running)'
!pgrep -af RunExperimentV6 || echo '(RunExperimentV6 not running - finished, not started, or between phases)'
print('--- throttle events so far ---')
!grep -c 'governor. THROTTLE' v6_run.log 2>/dev/null || echo 0
print('--- last 30 log lines ---')
!tail -n 30 v6_run.log 2>/dev/null || echo '(no log yet)'

### 6b. Thread-calibration curve
What the sweep measured, per budget level. `SustainedTurnsPerMin` is the second of two timed segments —
that is the number the knee is chosen on. `ThrottlePct` is how much slower the second segment was than
the first: a few percent is warm-up noise, a large positive value means the machine could not hold the
clock, and `ClockDropPct` (Linux only) shows the same thing directly.

In [ ]:
import pandas as pd, os
p = 'test_results/v6/_calibration/thread_calibration.csv'
if os.path.exists(p):
    t = pd.read_csv(p)
    cols = ['BudgetUnits','Threads','SustainedTurnsPerMin','ThrottlePct','SpeedupVs1',
            'EfficiencyPct','ClockDropPct','PeakHeapMB','PerTaskHeapMB','Chosen']
    display(t[cols].round(2))
    print('chosen per level:')
    display(t[t.Chosen == 'yes'][['BudgetUnits','Threads','SustainedTurnsPerMin','PerTaskHeapMB']])
else:
    print('no thread_calibration.csv yet - the first level has not been swept')

### 6c. Stopping the run
Use **SIGTERM, not `kill -9`**. SIGTERM is a cooperative stop: queued turns are abandoned, running
turns finish and write their rows, and no config is marked complete unless it really is — so nothing is
lost and section 5 resumes exactly where it stopped. `kill -9` skips all of that and costs the turns
that were in flight (at most one per worker); the resume still recovers, it just repeats those.

The lock is released on a clean stop, and a lock left by a `kill -9` goes stale on its own within two
minutes — no manual cleanup either way.

In [ ]:
# Graceful stop. Re-run section 5 afterwards to continue.
!pkill -TERM -f RunExperimentV6 && echo 'SIGTERM sent - watch v6_run.log for the drain banner' || echo '(nothing running)'

## 6d. Output integrity — run this before ANY analysis
Scans every leaf for the damage an interrupted or doubled run leaves: duplicated `(scenario, visit)`
turns, results rows with no telemetry row behind them, rows torn by a kill mid-write, and leaves marked
complete in `_completed.txt` that are not.

The duplicate case is the one to care about. Two writers on one tree — a backgrounded run that outlived
its kernel, plus a "resume" started beside it — produce a dataset with rows twice over. It does not
crash and it does not warn; it just doubles that turn's weight in every average downstream. The run
lock now prevents it, but this verifies the result, including for data produced before the lock existed.

`--repair` rewrites the affected leaves keeping the first complete copy of each turn (`.bak` kept), and
un-marks any leaf that is not actually finished so the next run completes it.

In [ ]:
!java -cp "out:lib/*" core.data.exp.VerifyV6Output test_results/v6

In [ ]:
# Only if the cell above reported problems:
# !java -cp "out:lib/*" core.data.exp.VerifyV6Output test_results/v6 --repair

## 7. Analyse (after the run completes)
`analyze_v6_full.py` imports `analyze_v5_full` and re-points it at the v6 tree, so **every v5 table and
figure is produced by identical code** (T1–T8, fig3–fig8, E1–E16, and the hypothesis battery E17–E21) —
which is what makes a v5-vs-v6 comparison meaningful. Two v6 data fixes are applied in the loader:
`FmCalls` is remapped to search-only calls (raw total kept as `FmCallsTotal`) so inherited cost metrics
are not inflated by the excluded greedy initialization, and the per-burn `ComputeUnitsElapsed` is dropped
at turn level so it cannot be silently averaged.

Then the new budget-adherence section:

| table | content |
|---|---|
| `B1_budget_adherence` | realized `SearchUnits` vs target per agent per level: mean, SD, overshoot, within-turn pacing |
| `B2_op_mix` | advances/copies/evals per turn, and what an equal-fmcalls budget *would* have given each agent |
| `B3_reference_conversion` | the C0 units→seconds table, split by config and by scenario source |
| `B4_init_cost` | `InitUnits` by source and BF tertile, and its share of the budget |

See `analysis/BUDGET_SUPPLEMENT.md` for the paper-facing write-up of what these establish.

In [ ]:
!python3 analysis/analyze_v6_full.py test_results/v6

## 8. Significance — the inferential answer
`analyze_v6_full.py` describes the data; this tests it, and it is what the write-up should quote.

Two things it does that the descriptive tables do not:

1. **Clusters by scenario.** The descriptive CI pools all ~10,000 turns per cell as if independent, but
   the 50 visits of a scenario share a board, a position and a greedy root. The independent replicate is
   the scenario (n=200). Pooling understates the standard error by up to sqrt(50) ~ 7x, at which point
   every difference looks significant. Visits are averaged first, then a paired Wilcoxon signed-rank
   with Benjamini-Hochberg correction, plus Cliff's delta so effect size is separate from significance.
2. **Reads the design as the 2x2 factorial it is.** `EMCTS` / `CrossoverBB` / `CrossoverExpansion@r` /
   `CrossoverCombo@r` are the four cells of BB-crossover x expansion-crossover. That identifies the
   **BB main effect**, the **expansion main effect**, and their **interaction** — whether the two
   mechanisms reinforce each other or get in each other's way. Comparing each config to EMCTS, which is
   all the descriptive analysis does, cannot separate those.

It also re-checks that each operator actually fired, so a cell where crossover never triggered is not
silently read as "crossover at this ratio does nothing".

In [ ]:
!python3 analysis/v6_significance.py test_results/v6
print(open('analysis/v6_significance/SIGNIFICANCE_V6.md').read())